# GNN for UML Diagram Pattern Recognition

This notebook implements a Graph Neural Network to recognize patterns in UML use case diagrams.

## Features:
- Parses PlantUML files into graph representations
- Builds PyTorch Geometric graphs with node features
- Trains GCN or GAT models for pattern classification
- Generates embeddings for similarity analysis

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

In [ ]:
import os
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.data import Data, DataLoader

## Part 1: UML Parsing Functions (Your Original Code)

In [ ]:
def parse_plantuml_raw(puml_path):
    """Parse PlantUML file and extract nodes and edges"""
    nodes = {}
    edges = []

    actor_pat   = re.compile(r'actor\s+"?(.+?)"?(\s+as\s+(\w+))?', re.I)
    usecase_pat = re.compile(r'usecase\s+"?(.+?)"?(\s+as\s+(\w+))?', re.I)
    class_pat   = re.compile(r'class\s+"?(.+?)"?(\s+as\s+(\w+))?', re.I)
    edge_pat    = re.compile(r'(.+?)\s*[-.]+[<>]*\s*(.+)')

    with open(puml_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("@"):
                continue

            m = actor_pat.match(line)
            if m:
                nodes[m.group(3) or m.group(1)] = "actor"
                continue

            m = usecase_pat.match(line)
            if m:
                nodes[m.group(3) or m.group(1)] = "usecase"
                continue

            m = class_pat.match(line)
            if m:
                nodes[m.group(3) or m.group(1)] = "class"
                continue

            m = edge_pat.match(line)
            if m:
                src = m.group(1).strip().strip('"')
                tgt = m.group(2).strip().strip('"')
                edges.append((src, tgt))
                nodes.setdefault(src, "usecase")
                nodes.setdefault(tgt, "usecase")

    return nodes, edges

In [ ]:
def extract_action(name):
    """Extract action verb from use case name"""
    name = name.lower()
    tokens = re.findall(r"[a-z]+", name)
    if not tokens:
        return None

    action = tokens[0]

    if len(action) < 4:
        return None
    if action.endswith(("service", "system")):
        return None
    if action in ("user", "customer", "patient", "admin", "manager"):
        return None

    return action

In [ ]:
def build_action_order(all_usecase_names):
    """Build frequency-based action ordering"""
    freq = Counter()
    for name in all_usecase_names:
        action = extract_action(name)
        if action:
            freq[action] += 1
    return [a for a, _ in freq.most_common()]

In [ ]:
def rank(name, action_order):
    """Rank action by frequency"""
    name = name.lower()
    for i, action in enumerate(action_order):
        if action in name:
            return i
    return len(action_order)

In [ ]:
def correct_edges(nodes, edges, action_order):
    """Correct edge directions based on node types and action ordering"""
    corrected = []

    for src, tgt in edges:
        s, t = nodes.get(src), nodes.get(tgt)

        if s == "actor" and t == "usecase":
            corrected.append((src, tgt))

        elif s == "usecase" and t == "actor":
            corrected.append((tgt, src))

        elif s == "usecase" and t == "usecase":
            if rank(src, action_order) <= rank(tgt, action_order):
                corrected.append((src, tgt))
            else:
                corrected.append((tgt, src))

        elif s == "usecase" and t == "class":
            corrected.append((src, tgt))

        elif s == "class" and t == "usecase":
            corrected.append((tgt, src))

    return corrected

In [ ]:
def collect_all_usecases(dataset_root):
    """Collect all use cases from dataset"""
    all_usecases = []

    for root, _, files in os.walk(dataset_root):
        for file in files:
            if file.endswith(".puml"):
                nodes, _ = parse_plantuml_raw(os.path.join(root, file))
                for name, t in nodes.items():
                    if t == "usecase":
                        all_usecases.append(name)

    return all_usecases

In [ ]:
def parse_and_correct_plantuml(puml_path, action_order):
    """Parse and correct a single PlantUML file"""
    nodes, raw_edges = parse_plantuml_raw(puml_path)
    edges = correct_edges(nodes, raw_edges, action_order)

    uml = {
        "actors": [],
        "usecases": [],
        "classes": [],
        "relations": []
    }

    type_map = {
        "actor": "actors",
        "usecase": "usecases",
        "class": "classes"
    }

    id_map = {}
    counter = 0

    def get_id(name):
        nonlocal counter
        if name not in id_map:
            id_map[name] = f"N{counter}"
            counter += 1
        return id_map[name]

    for name, t in nodes.items():
        if t in type_map:
            uml[type_map[t]].append((get_id(name), name))

    for src, tgt in edges:
        uml["relations"].append((get_id(src), get_id(tgt)))

    return uml

In [ ]:
def parse_and_correct_dataset(dataset_root, action_order):
    """Parse entire dataset"""
    corrected_models = []

    for root, _, files in os.walk(dataset_root):
        for file in files:
            if file.endswith(".puml"):
                uml = parse_and_correct_plantuml(
                    os.path.join(root, file),
                    action_order
                )
                corrected_models.append(uml)

    return corrected_models

## Part 2: Graph Construction for GNN

In [ ]:
class UMLGraphBuilder:
    """Builds PyTorch Geometric graphs from UML diagrams"""
    
    def __init__(self):
        self.node_type_encoder = LabelEncoder()
        self.action_encoder = LabelEncoder()
        self.vocabulary = {}
        self.vocab_size = 0
        
    def fit_encoders(self, all_models):
        """Fit label encoders on all models"""
        all_types = []
        all_actions = []
        all_names = []
        
        for model in all_models:
            # Collect node types
            for node_list in [model['actors'], model['usecases'], model['classes']]:
                for _, name in node_list:
                    all_names.append(name)
                    
            all_types.extend(['actor'] * len(model['actors']))
            all_types.extend(['usecase'] * len(model['usecases']))
            all_types.extend(['class'] * len(model['classes']))
            
            # Collect actions from use cases
            for _, name in model['usecases']:
                action = extract_action(name)
                if action:
                    all_actions.append(action)
        
        # Fit encoders
        self.node_type_encoder.fit(all_types)
        if all_actions:
            self.action_encoder.fit(all_actions)
        
        # Build vocabulary for node names
        unique_names = list(set(all_names))
        self.vocabulary = {name: idx for idx, name in enumerate(unique_names)}
        self.vocab_size = len(unique_names)
        
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Node types: {list(self.node_type_encoder.classes_)}")
        print(f"Actions: {len(all_actions)} unique actions")
        
    def create_node_features(self, model):
        """Create feature vectors for each node"""
        features = []
        node_to_idx = {}
        idx = 0
        
        # Process all nodes
        for node_type, node_list_key in [('actor', 'actors'), 
                                          ('usecase', 'usecases'), 
                                          ('class', 'classes')]:
            for node_id, name in model[node_list_key]:
                node_to_idx[node_id] = idx
                
                # Feature vector: [node_type_encoding, action_encoding, name_vocab_idx, name_length]
                type_enc = self.node_type_encoder.transform([node_type])[0]
                
                # Action encoding (for use cases)
                action = extract_action(name) if node_type == 'usecase' else None
                if action and action in self.action_encoder.classes_:
                    action_enc = self.action_encoder.transform([action])[0]
                else:
                    action_enc = -1
                
                # Name vocabulary index
                name_idx = self.vocabulary.get(name, -1)
                
                # Create feature vector
                feature = [type_enc, action_enc, name_idx, len(name)]
                features.append(feature)
                
                idx += 1
                
        return torch.tensor(features, dtype=torch.float), node_to_idx
    
    def create_edge_index(self, model, node_to_idx):
        """Create edge index for PyG"""
        edges = []
        
        for src_id, tgt_id in model['relations']:
            if src_id in node_to_idx and tgt_id in node_to_idx:
                src_idx = node_to_idx[src_id]
                tgt_idx = node_to_idx[tgt_id]
                edges.append([src_idx, tgt_idx])
        
        if edges:
            return torch.tensor(edges, dtype=torch.long).t().contiguous()
        else:
            # Empty graph case
            return torch.zeros((2, 0), dtype=torch.long)
    
    def build_graph(self, model, label=None):
        """Build a PyTorch Geometric graph from UML model"""
        node_features, node_to_idx = self.create_node_features(model)
        edge_index = self.create_edge_index(model, node_to_idx)
        
        data = Data(
            x=node_features,
            edge_index=edge_index,
            num_nodes=len(node_to_idx)
        )
        
        if label is not None:
            data.y = torch.tensor([label], dtype=torch.long)
        
        return data

## Part 3: GNN Model Architectures

In [ ]:
class GCN_UML_Classifier(nn.Module):
    """Graph Convolutional Network for UML pattern classification"""
    
    def __init__(self, input_dim, hidden_dim=64, num_classes=6, dropout=0.5):
        super(GCN_UML_Classifier, self).__init__()
        
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, num_classes)
        
        self.dropout = dropout
        
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # Graph convolution layers
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        
        # Global pooling
        x = global_mean_pool(x, batch)
        
        # Classification layers
        x = self.fc1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.fc2(x)
        
        return F.log_softmax(x, dim=1)

In [ ]:
class GAT_UML_Classifier(nn.Module):
    """Graph Attention Network for UML pattern classification"""
    
    def __init__(self, input_dim, hidden_dim=64, num_classes=6, heads=4, dropout=0.5):
        super(GAT_UML_Classifier, self).__init__()
        
        self.conv1 = GATConv(input_dim, hidden_dim, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, dropout=dropout)
        
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, num_classes)
        
        self.dropout = dropout
        
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # Graph attention layers
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = F.elu(x)
        
        # Global pooling
        x = global_mean_pool(x, batch)
        
        # Classification layers
        x = self.fc1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.fc2(x)
        
        return F.log_softmax(x, dim=1)

## Part 4: Training Pipeline

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def evaluate(model, loader, device):
    """Evaluate model"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            pred = out.argmax(dim=1)
            
            correct += (pred == batch.y).sum().item()
            total += batch.y.size(0)
    
    return correct / total if total > 0 else 0

## Part 5: Load and Prepare Data

In [ ]:
# Mount Google Drive (if using Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set dataset path
DATASET_PATH = "/content/drive/MyDrive/uml_dataset"

# Parse dataset
print("Collecting use cases...")
all_usecases = collect_all_usecases(DATASET_PATH)
action_order = build_action_order(all_usecases)

print("Learned Action Order:")
print(action_order)

print("\nParsing dataset...")
corrected_models = parse_and_correct_dataset(DATASET_PATH, action_order)
print(f"Total UML models: {len(corrected_models)}")

In [ ]:
# Create labels based on directory structure
# Assuming directory names correspond to pattern types
labels = []
pattern_names = []

for root, _, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".puml"):
            # Extract pattern name from directory
            pattern = os.path.basename(root)
            pattern_names.append(pattern)

# Encode pattern names as labels
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(pattern_names)

print(f"Pattern categories: {list(label_encoder.classes_)}")
print(f"Number of samples per category:")
for i, cat in enumerate(label_encoder.classes_):
    count = (labels == i).sum()
    print(f"  {cat}: {count}")

## Part 6: Build Graphs

In [ ]:
# Initialize graph builder
graph_builder = UMLGraphBuilder()

# Fit encoders
print("Fitting encoders...")
graph_builder.fit_encoders(corrected_models)

# Build graphs
print("\nBuilding graphs...")
graphs = []
for i, model in enumerate(corrected_models):
    graph = graph_builder.build_graph(model, labels[i])
    graphs.append(graph)

print(f"Created {len(graphs)} graphs")
print(f"Example graph: {graphs[0]}")

## Part 7: Train-Test Split

In [ ]:
# Split data
train_graphs, test_graphs = train_test_split(
    graphs, 
    test_size=0.2, 
    random_state=42,
    stratify=labels
)

train_graphs, val_graphs = train_test_split(
    train_graphs,
    test_size=0.2,
    random_state=42
)

print(f"Train: {len(train_graphs)}, Val: {len(val_graphs)}, Test: {len(test_graphs)}")

# Create data loaders
train_loader = DataLoader(train_graphs, batch_size=8, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=8)
test_loader = DataLoader(test_graphs, batch_size=8)

## Part 8: Initialize and Train Model

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model parameters
input_dim = graphs[0].x.shape[1]
hidden_dim = 64
num_classes = len(label_encoder.classes_)

print(f"\nModel configuration:")
print(f"  Input dim: {input_dim}")
print(f"  Hidden dim: {hidden_dim}")
print(f"  Num classes: {num_classes}")

In [ ]:
# Initialize GCN model
model = GCN_UML_Classifier(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    dropout=0.5
).to(device)

# Optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
criterion = nn.NLLLoss()

print(model)

In [ ]:
# Training loop
epochs = 100
train_losses = []
val_accuracies = []
best_val_acc = 0

print("Starting training...\n")

for epoch in range(epochs):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(train_loss)
    
    # Validate
    val_acc = evaluate(model, val_loader, device)
    val_accuracies.append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_uml_gnn_model.pt')
    
    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}')

print(f"\nTraining complete! Best validation accuracy: {best_val_acc:.4f}")

## Part 9: Evaluate and Visualize Results

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_uml_gnn_model.pt'))

# Test accuracy
test_acc = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

# Accuracy curve
ax2.plot(val_accuracies)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get predictions
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch)
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, 
                          target_names=label_encoder.classes_))

## Part 10: Test on New UML Diagram

In [ ]:
def predict_uml_pattern(puml_path, model, graph_builder, label_encoder, action_order, device):
    """Predict pattern for a new UML diagram"""
    # Parse UML
    uml_model = parse_and_correct_plantuml(puml_path, action_order)
    
    # Build graph
    graph = graph_builder.build_graph(uml_model)
    graph = graph.to(device)
    
    # Add batch dimension
    graph.batch = torch.zeros(graph.num_nodes, dtype=torch.long, device=device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        out = model(graph)
        pred = out.argmax(dim=1)
        confidence = torch.exp(out[0, pred]).item()
    
    pattern = label_encoder.inverse_transform([pred.item()])[0]
    
    return pattern, confidence


# Example: Predict on a test file
test_file = "/path/to/test.puml"
pattern, conf = predict_uml_pattern(test_file, model, graph_builder, 
                                     label_encoder, action_order, device)
print(f"Predicted pattern: {pattern} (confidence: {conf:.4f})")